In [77]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from datetime import datetime

file_path = 'vdjdb_structures_annotated.tsv'

df = pd.read_csv(file_path, sep='\t', low_memory=False)
has_structure = df['ranking_confidence'].notna()
struct_df = df[has_structure].copy()

total_records = len(df)
num_structures = len(struct_df)
pct = num_structures / total_records * 100

mean_rank = struct_df['ranking_confidence'].mean()
mean_contacts = struct_df['num_contacts'].mean()
mean_scanning = struct_df['scanning_angle'].mean()
mean_pitch = struct_df['pitch_angle'].mean()

epitope_counts = df['antigen.epitope'].value_counts()
top15 = epitope_counts.head(15).index.tolist()

stats_list = []
for epi in top15:
    epi_df = df[df['antigen.epitope'] == epi]
    total = len(epi_df)
    structures = epi_df['ranking_confidence'].notna().sum()
    stats_list.append({
        'Epitope': epi,
        'Total Records': total,
        'Structures': structures,
        '% Structured': round(structures / total * 100, 2) if total > 0 else 0,
        'Mean Ranking Confidence': round(epi_df['ranking_confidence'].mean(), 4) if structures > 0 else None,
        'Mean Contacts': round(epi_df['num_contacts'].mean(), 2) if structures > 0 else None
    })

other_df = df[~df['antigen.epitope'].isin(top15)]
other_total = len(other_df)
other_struct = other_df['ranking_confidence'].notna().sum()
stats_list.append({
    'Epitope': 'OTHER (rest)',
    'Total Records': other_total,
    'Structures': other_struct,
    '% Structured': round(other_struct / other_total * 100, 2) if other_total > 0 else 0,
    'Mean Ranking Confidence': round(other_df['ranking_confidence'].mean(), 4) if other_struct > 0 else None,
    'Mean Contacts': round(other_df['num_contacts'].mean(), 2) if other_struct > 0 else None
})

stats_df = pd.DataFrame(stats_list)

bar_df = stats_df[
    (stats_df['Structures'] > 0) & 
    (stats_df['Epitope'] != 'OTHER (rest)')
].sort_values('Structures', ascending=False)

fig_rank = px.histogram(struct_df, x='ranking_confidence', nbins=50)
fig_contacts = px.histogram(struct_df, x='num_contacts', nbins=60)

fig_epitopes = px.bar(
    bar_df,
    x='Epitope',
    y='Structures',
    color='Mean Ranking Confidence',
    color_continuous_scale='Viridis',
    range_color=[0, 1],
    hover_data=['Total Records', '% Structured', 'Mean Contacts'],
    category_orders={"Epitope": bar_df['Epitope'].tolist()}
)
fig_epitopes.update_layout(xaxis_tickangle=-45, height=650)

fig_angles = px.scatter(
    struct_df, x='scanning_angle', y='pitch_angle',
    color='ranking_confidence',
    color_continuous_scale='Viridis',
    range_color=[0, 1],
    hover_data=['num_contacts', 'antigen.epitope']
)

fig_summary = make_subplots(rows=2, cols=2, subplot_titles=("Ranking Confidence", "Contacts", "Scanning Angle", "Pitch Angle"))
cols = ['ranking_confidence', 'num_contacts', 'scanning_angle', 'pitch_angle']
for i, col in enumerate(cols):
    fig_summary.add_trace(go.Histogram(x=struct_df[col], nbinsx=40, name=""), row=i//2+1, col=i%2+1)

fig_summary.update_layout(height=700, showlegend=False)
fig_summary.update_traces(showlegend=False)

html_content = f"""<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <title>VDJdb Structures Report</title>
    <script src="https://cdn.plot.ly/plotly-2.35.2.min.js"></script>
    <style>
        @import url('https://fonts.googleapis.com/css2?family=Inter:wght@400;500;600&amp;display=swap');
        
        body {{
            font-family: Inter, -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, "Helvetica Neue", Arial, sans-serif;
            margin: 40px 20px;
            background: #f8f9fa;
            line-height: 1.6;
            color: #1f2937;
        }}
        .container {{
            max-width: 1200px;
            margin: 0 auto;
            background: white;
            padding: 40px 50px;
            border-radius: 16px;
            box-shadow: 0 10px 30px rgba(0,0,0,0.08);
        }}
        h1 {{
            font-size: 28px;
            font-weight: 600;
            margin-bottom: 8px;
            letter-spacing: -0.02em;
        }}
        h2 {{
            font-size: 20px;
            font-weight: 600;
            margin-top: 40px;
            margin-bottom: 12px;
            color: #111827;
        }}
        hr {{
            border: none;
            border-top: 1px solid #e2e8f0;
            margin: 24px 0;
        }}
        .summary {{
            display: grid;
            grid-template-columns: repeat(auto-fit, minmax(260px, 1fr));
            gap: 20px;
            margin: 32px 0;
        }}
        .card {{
            background: #f8fafc;
            padding: 24px;
            border-radius: 12px;
            box-shadow: 0 4px 12px rgba(0,0,0,0.05);
            border: 1px solid #e2e8f0;
        }}
        .card h3 {{
            font-size: 14px;
            font-weight: 500;
            color: #64748b;
            margin-bottom: 8px;
        }}
        .card h2 {{
            font-size: 26px;
            font-weight: 600;
            margin: 0;
            color: #0f172a;
        }}
        table {{
            width: 100%;
            border-collapse: collapse;
            margin: 24px 0;
            font-size: 14px;
        }}
        th, td {{
            padding: 14px 16px;
            border-bottom: 1px solid #e2e8f0;
            text-align: left;
        }}
        th {{
            background: #f1f5f9;
            font-weight: 600;
            color: #475569;
        }}
        tr:hover {{
            background: #f8fafc;
        }}
        .note {{
            font-size: 13px;
            color: #64748b;
        }}
    </style>
</head>
<body>
    <div class="container">
        <h1>VDJdb Structures</h1>
        <p class="note">Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')} &nbsp;|&nbsp; Total complete nonredundant records: {total_records:,} &nbsp;|&nbsp; Unique epitopes: {df['antigen.epitope'].nunique():,}</p>

        <div class="summary">
            <div class="card">
                <h3>Records with structures</h3>
                <h2>{num_structures:,} <span style="font-size:18px; font-weight:500; color:#64748b;">({pct:.1f}%)</span></h2>
            </div>
            <div class="card">
                <h3>Mean ranking confidence</h3>
                <h2>{mean_rank:.4f}</h2>
            </div>
            <div class="card">
                <h3>Mean contacts</h3>
                <h2>{mean_contacts:.1f}</h2>
            </div>
            <div class="card">
                <h3>Mean angles</h3>
                <p style="margin:0; font-size:15px;">
                    <strong>Scanning:</strong> {mean_scanning:.1f}° &nbsp;&nbsp;
                    <strong>Pitch:</strong> {mean_pitch:.1f}°
                </p>
            </div>
        </div>

        <h2>Ranking Confidence</h2>
        <hr>
        <div id="rank_plot"></div>

        <h2>Number of Contacts</h2>
        <hr>
        <div id="contacts_plot"></div>

        <h2>Top 15 Epitopes by Number of Structures</h2>
        <hr>
        <div id="epitopes_plot"></div>

        <h2>Scanning vs Pitch Angle</h2>
        <hr>
        <div id="angles_plot"></div>

        <h2>All Metrics Histograms</h2>
        <hr>
        <div id="summary_plot"></div>

        <h2>Epitopes-structures features</h2>
        <hr>
        {stats_df.to_html(index=False, escape=False)}

        <script>
            Plotly.newPlot("rank_plot", {fig_rank.to_json()}, {{responsive: true}});
            Plotly.newPlot("contacts_plot", {fig_contacts.to_json()}, {{responsive: true}});
            Plotly.newPlot("epitopes_plot", {fig_epitopes.to_json()}, {{responsive: true}});
            Plotly.newPlot("angles_plot", {fig_angles.to_json()}, {{responsive: true}});
            Plotly.newPlot("summary_plot", {fig_summary.to_json()}, {{responsive: true}});
        </script>
    </div>
</body>
</html>
"""

with open('vdjdb_report_final.html', 'w', encoding='utf-8') as f:
    f.write(html_content)


✅ Report saved as vdjdb_report_final.html
   • No plot titles (removed duplication)
   • No section numbers
   • Clean horizontal lines between sections


In [72]:
zero_contacts = vdjdb_structures_annotated[vdjdb_structures_annotated.num_contacts == 0]
zero_contacts

,cdr3.alpha,v.alpha,j.alpha,cdr3.beta,v.beta,d.beta,j.beta,species,mhc.a,mhc.b,...,vdjdb.score,TCR_hash,ranking_confidence,plddt,ptm,iptm,tcr-pmhc_iptm,num_contacts,scanning_angle,pitch_angle
484,CAASIGNFGNEKLTF,TRAV23/DV6*01,TRAJ48*01,CASSPSRNTEAFF,TRBV4-3*01,NaN,TRBJ1-1*01,HomoSapiens,HLA-B*07:02,B2M,...,0,a137dc02a7eba80dfbfb45c5c4d6d4d0551f05ec136859...,0.526179,87.054748,0.624198,0.501675,0.332093,0.0,NaN,NaN
486,CAASVGNFGNEKLTF,TRAV23/DV6*01,TRAJ48*01,CASSPYRNTEAFF,TRBV4-3*01,NaN,TRBJ1-1*01,HomoSapiens,HLA-B*07:02,B2M,...,0,4da647a1f10afe067ff61f591ea2f10e08ba1d9e26e512...,0.706266,90.550063,0.762929,0.692100,0.602699,0.0,NaN,NaN
563,CVMGWGKLQF,TRAV3*01,TRAJ24*01,CASSLDRSTGELFF,TRBV27*01,NaN,TRBJ2-2*01,HomoSapiens,HLA-A*02:01,B2M,...,0,01e0353b19ba37292d012e9883b157376b5058d295d7fe...,0.815843,93.618624,0.849303,0.807479,0.749061,0.0,NaN,NaN
565,CAVRLNFGNEKLTF,TRAV41*01,TRAJ48*01,CAWSVWQMSSGANVLTF,TRBV30*01,NaN,TRBJ2-6*01,HomoSapiens,HLA-A*02:01,B2M,...,0,858f9b24b3578cfd8cd524e31b6a460ffdd523aef1b2c4...,0.706968,89.784219,0.779682,0.688789,0.649569,0.0,NaN,NaN
643,CGTERPNDYKLSF,TRAV30*01,TRAJ20*01,CASSTGQLSNTGELFF,TRBV9*01,NaN,TRBJ2-2*01,HomoSapiens,HLA-A*02:01,B2M,...,0,04c62fa83f6f13c8c5eee6dd41659c6fbe16de191e3049...,0.905018,94.377947,0.918703,0.901596,0.894729,0.0,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
74452,CAMSISNFGNEKLTF,TRAV12-3*01,TRAJ48*01,CASSDTGTSGFEAKNIQYF,TRBV9*01,NaN,TRBJ2-4*01,HomoSapiens,HLA-A*02:01,B2M,...,0,8cfd25bf2b98388f20f81f08b68101ad72d93192804cea...,0.804319,92.128140,0.839639,0.795489,0.743415,0.0,NaN,NaN
74469,CASLTPGASTIIF,TRAV8-6*01,TRAJ3*01,CASSSGLVAPGELFF,TRBV2*01,NaN,TRBJ2-2*01,HomoSapiens,HLA-A*02:01,B2M,...,0,9d07f41a53735e21b1faf9d1b43d7a93df3f9d2a8d8452...,0.641623,85.561658,0.716661,0.622864,0.520376,0.0,NaN,NaN
74492,CATAISNFGNEKLTF,TRAV12-3*01,TRAJ48*01,CASSSTSKDGILAKNIQYF,TRBV5-1*01,NaN,TRBJ2-4*01,HomoSapiens,HLA-A*02:01,B2M,...,0,d34f4f617b428ae7f7962551a97c49fd4a98a5783b0ee9...,0.903273,94.847320,0.922533,0.898459,0.894736,0.0,NaN,NaN
74543,CAMSISNFGNEKLPF,TRAV12-3*01,TRAJ48*01,CASSFEDRVTNYGYTF,TRBV5-1*01,NaN,TRBJ1-2*01,HomoSapiens,HLA-A*02:01,B2M,...,0,642b1cd38f7ec4de38a665e747bd062f7e8d9f66c8f02a...,0.898912,94.585217,0.918978,0.893895,0.878318,0.0,NaN,NaN


In [74]:
zero_contacts['antigen.epitope'].value_counts()

antigen.epitope
NLVPMVATV      348
KLGGALQAK      191
GLCTLVAML       13
YVLDHLIVV        9
GILGFVFTL        7
LLAGIGTVPI       5
RAKFKQLL         5
YLQPRTFLL        3
RPHERNGFTVL      2
PTDNYITTY        2
AVFDRKSDAK       2
IVTDFSVIK        1
RPPIFIRRL        1
TTDPSFLGRY       1
RLRAEAQVK        1
FLYALALLL        1
FLRGRAYGL        1
Name: count, dtype: int64

In [76]:
zero_contacts['vdjdb.score'].value_counts()

vdjdb.score
0    575
2     12
3      3
1      3
Name: count, dtype: int64

In [75]:
zero_contacts['mhc.a'].value_counts()

mhc.a
HLA-A*02:01    386
HLA-A*03:01    192
HLA-B*08:01      6
HLA-B*07:02      3
HLA-A*01:01      3
HLA-A*11:01      3
Name: count, dtype: int64

In [ ]:
zero_contacts['antigen.epitope'].value_counts()